# Fixed-point arithmetic with a rotation-based Boltzmann coin

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$

`FixedPointArithmetic` uses the same coefficient loading, energy-difference computation, and positive-part selector as `HybridPhaseArithmetic`. It replaces the uphill QSP branch with independent controlled rotations. The hybrid implementation remains available in `3.2_hybrid_arithmetic.ipynb`.

For the represented Hamiltonian, $z=\max\{0,H(y)-H(x)\}$ and $A=e^{-\beta z}$. Acceptance is the all-zero state of the complete coin register.

## Imports and fixed-point convention

Run from the repository's `notebooks` directory with the updated `monaqa2` package beside it. The numerical and resource examples below use NumPy and the runtime API; circuit examples additionally require Qiskit, and the full-arithmetic simulation requires Qiskit Aer.

In [ ]:
import sys

if ".." not in sys.path:
    sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt

from monaqa2.qiskit.utils_numpy import boltzmann_rotation_angles
from monaqa2.data.runtime import fixed_point_arithmetic_circuit

The API uses $F$ fractional bits and a little-endian signed Q1.F word of width $W=F+1$. Coefficients are normalized by $\alpha=2(\sum_i|h_i|+\sum_{i<j}|J_{ij}|)$ and rounded to nearest even, as in the hybrid implementation. After the positive-part selector,

$$z=\alpha\sum_{j=0}^{W-1}z_j2^{j-F},\qquad z_j\in\{0,1\}.$$

The former sign bit is zero on valid nonnegative inputs. One clean coin ancilla is allocated for each of the $W$ bits, with

$$\cos(\theta_j/2)=\exp[-\beta\alpha2^{j-F}/2].$$

The accepting amplitude is the product of the active cosines, $e^{-\beta z/2}$. No polynomial, cutoff, QSP sequence, or exponentiation circuit is used.

In [ ]:
F, beta, alpha = 5, 2.0, 3.0
W = F + 1
angles = boltzmann_rotation_angles(W, beta, alpha)
words = np.arange(1 << F)  # All nonnegative words; the signed high bit is zero.
bits = (words[:, None] >> np.arange(W)) & 1
energies = alpha * words / (1 << F)
probabilities = np.prod(np.where(bits, np.cos(angles / 2) ** 2, 1.0), axis=1)
expected = np.exp(-beta * energies)
np.testing.assert_allclose(probabilities, expected, atol=2e-14, rtol=2e-14)

print(" j | physical bit weight | theta")
for j, theta in enumerate(angles):
    print(f"{j:2d} | {alpha * 2.0 ** (j - F):19.8f} | {theta:.10f}")
print("maximum probability error:", np.max(np.abs(probabilities - expected)))

The product-state calculation agrees with the Metropolis probability at every representable nonnegative word.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(energies, expected, label="Metropolis")
ax.scatter(energies, probabilities, s=15, label="rotation coin")
ax.set(xlabel="Positive energy difference", ylabel="Acceptance probability")
ax.legend()
plt.show()

## Circuit API

`RotationBoltzmannCoin(W, beta, alpha)` acts on `delta | coins`. `FixedPointArithmetic(n, h, J, F, beta)` adds the reversible energy computation and its uncomputation. Its layout is `coins | x | y | delta | zero | signs | work`. Initialize all registers other than `x` and `y` to zero; only the coins may remain populated after the gate.

In [ ]:
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import Statevector

from monaqa2.qiskit.arithmetic_fixed_point import FixedPointArithmetic, RotationBoltzmannCoin
from monaqa2.qiskit.utils_qiskit import get_nc_depth, qiskit_to_clifford_rz

coin = RotationBoltzmannCoin(W, beta, alpha)
for word in (0, 1, 2, 3, 5, (1 << F) - 1):
    qc = QuantumCircuit(coin.num_qubits)
    for j, q in enumerate(coin.layout["delta"]):
        if (word >> j) & 1:
            qc.x(q)
    qc.append(coin, range(coin.num_qubits))
    simulated = Statevector.from_instruction(qc).probabilities(coin.layout["coins"])[0]
    target = np.exp(-beta * alpha * word / (1 << F))
    np.testing.assert_allclose(simulated, target, atol=2e-12, rtol=2e-12)
    print(f"word={word:2d}: simulated={simulated:.10f}, expected={target:.10f}")

The following small Ising instance has energy differences inside the signed register range. Spin encoding is `s_j = (-1)**x_j`, with bit j on `layout["x"][j]`. The constructor has no `eps`, `degree`, or mocked-QSP options.

In [ ]:
n, F, beta = 2, 3, 0.7
h = np.array([0.5, -0.25])
J = np.array([[0.0, 0.25], [0.25, 0.0]])
arithmetic = FixedPointArithmetic(n, h, J, F, beta)
print("qubits:", arithmetic.num_qubits)
print("coin qubits:", len(arithmetic.layout["coins"]))
print("register sizes:", {name: len(wires) for name, wires in arithmetic.layout.items()})

## Acceptance and arithmetic cleanup

The full circuit allocates many scratch qubits, so use Aer matrix product states rather than a dense statevector. These coefficients are represented exactly at F=3. Check zero, negative, single-bit positive, and multiple-bit positive differences, together with cleanup of every arithmetic ancilla.

In [ ]:
from qiskit_aer import AerSimulator

backend = AerSimulator(method="matrix_product_state", matrix_product_state_truncation_threshold=1e-16)
layout = arithmetic.layout
clean = layout["delta"] + layout["zero"] + layout["signs"] + layout["work"]

def energy(word):
    spins = np.array([1 - 2 * ((word >> j) & 1) for j in range(n)])
    return float(h @ spins + J[0, 1] * spins[0] * spins[1])

for x, y in ((0, 0), (0, 1), (3, 0), (1, 0)):
    qc = QuantumCircuit(arithmetic.num_qubits)
    for register, word in ((layout["x"], x), (layout["y"], y)):
        for j, q in enumerate(register):
            if (word >> j) & 1:
                qc.x(q)
    qc.append(arithmetic, range(arithmetic.num_qubits))
    qc = qiskit_to_clifford_rz(qc)
    qc.save_probabilities(layout["coins"], label="coins")
    for q in clean:
        qc.save_probabilities([q], label=f"clean_{q}")
    result = backend.run(qc, shots=1).result()
    assert result.success
    data = result.data(0)
    delta = energy(y) - energy(x)
    expected = np.exp(-beta * max(delta, 0.0))
    np.testing.assert_allclose(data["coins"][0], expected, atol=2e-10, rtol=2e-10)
    for q in clean:
        np.testing.assert_allclose(data[f"clean_{q}"][0], 1.0, atol=2e-10, rtol=0)
    print(f"x={x}, y={y}, delta={delta:+.2f}, acceptance={data['coins'][0]:.10f}")

## Resources and walk integration

Let $M=n(n+1)/2$ and $s_{\mathrm{SK}}=\lceil\log_{3/2}(2M)\rceil$. The current resource model uses

| Block | Logical qubits | Non-Clifford depth bound |
|---|---:|---:|
| Independent rotation coin | $2W$ | $2$ |
| Complete fixed-point arithmetic | $2n+6MW-2M+W+1$ | $2(18s_{\mathrm{SK}}+18W)+6+D_{\mathrm{rot}}$ |
| Reflection | $2n+W+2$ | $14\lceil\log_2(n+W-1)\rceil-10$ |
| Accept path | $3n+W$ | $28\lceil\log_2W\rceil-11$ |

Here $D_{\mathrm{rot}}=2$ at positive beta and zero at beta=0; reflection and accept-path rows assume $n\geq3$, $W\geq2$. A controlled Ry uses two sequential Rz rotations and all bit/coin pairs run in parallel. The model counts each arbitrary Rz as one non-Clifford layer and does not include discrete Clifford+T synthesis or factory costs.

`monaqa2.data.runtime` now supports `arithmetic_type="FIXED_POINT"`. Its public `fixed_point_arithmetic_circuit` chooses F from the retained runtime precision rule, so its F need not match the small explicit circuit above. Pass `coins=len(arithmetic.layout["coins"])` to `Reflection` and `AcceptPath`; the complete walk wiring is shown in `3_quantum_walks.ipynb`.

In [ ]:
depth, qubits = fixed_point_arithmetic_circuit(n=50, eps_W=1e-10, beta=4.0)
print("estimated fixed-point arithmetic depth:", depth)
print("estimated fixed-point arithmetic qubits:", qubits)

The rotations implement the exact probability for the represented energy difference, up to angle precision and subsequent gate synthesis. Coefficient rounding and fixed-width modular overflow are inherited from the hybrid arithmetic: normalized signed differences must fit $[-1,1-2^{-F}]$, and the positive endpoint +1 is not representable. The examples above avoid that endpoint.